In [0]:
-- 1. Read, Type, Trim, and generate Surrogate Key (Streaming)
-- Note: DISTINCT is removed because APPLY CHANGES INTO natively handles deduplication on the primary key.
CREATE TEMPORARY STREAMING TABLE stg_promotions_typed AS
SELECT 
  -- Generate a unique Surrogate Key for this specific version of the promotion record
  md5(concat_ws('|', TRIM(promotion_id), CAST(ingested_timestamp AS STRING))) AS promotion_sk,
  
  CAST(TRIM(promotion_id) AS INT) AS promotion_id,
  TRIM(promotion_name) AS promotion_name,
  TRIM(promotion_type) AS promotion_type,
  CAST(TRIM(discount_percent) AS DOUBLE) AS discount_percent,
  
  -- Handle malformed start_date strings
  COALESCE(
    try_to_date(TRIM(start_date), 'M/d/yyyy'),
    try_to_date(TRIM(start_date), 'yyyy-M-d'),
    try_to_date(TRIM(start_date), 'd-MMM-yy'),
    CAST(TRIM(start_date) AS DATE)
  ) AS start_date,
  
  -- Apply same robust casting to end_date
  COALESCE(
    try_to_date(TRIM(end_date), 'M/d/yyyy'),
    try_to_date(TRIM(end_date), 'yyyy-M-d'),
    try_to_date(TRIM(end_date), 'd-MMM-yy'),
    CAST(TRIM(end_date) AS DATE)
  ) AS end_date,
  
  TRIM(channel) AS channel,
  TRIM(media_type) AS media_type,
  
  ingested_timestamp,
  batch_date
FROM STREAM(mavenmarket.bronze.promotions);

-- 2. Run Generic KPI Checks (Completeness, Validity, Consistency)
CREATE TEMPORARY STREAMING TABLE stg_promotions_checked AS
SELECT 
  *,
  -- HARD CHECKS (Route to Quarantine)
  CASE WHEN promotion_id IS NULL THEN 'Completeness: promotion_id is null; ' ELSE '' END AS err_promo_id,
  CASE WHEN discount_percent < 0 THEN 'Validity: discount_percent is negative; ' ELSE '' END AS err_discount_neg,
  CASE WHEN discount_percent > 1.0 THEN 'Validity: discount_percent exceeds 100%; ' ELSE '' END AS err_discount_high,
  CASE WHEN end_date < start_date THEN 'Consistency: end_date is before start_date; ' ELSE '' END AS err_dates,
  
  -- WARN CHECKS (Stay in Silver, added to dq_flags)
  CASE WHEN promotion_name IS NULL THEN 'Completeness: promotion_name is null; ' ELSE '' END AS warn_name,
  CASE WHEN channel IS NULL THEN 'Completeness: channel is null; ' ELSE '' END AS warn_channel
FROM STREAM(LIVE.stg_promotions_typed);

-- 3. Consolidate Flags
CREATE TEMPORARY STREAMING TABLE stg_promotions_flagged AS
SELECT 
  *,
  TRIM(CONCAT(err_promo_id, err_discount_neg, err_discount_high, err_dates)) AS hard_failures,
  TRIM(CONCAT(warn_name, warn_channel)) AS dq_flags
FROM STREAM(LIVE.stg_promotions_checked);

-- 4. Quarantine Table Setup (Records failing hard checks)
CREATE STREAMING TABLE quarantine_promotions
COMMENT "Quarantined promotions failing hard checks"
AS SELECT 
  * EXCEPT(err_promo_id, err_discount_neg, err_discount_high, err_dates, warn_name, warn_channel, dq_flags, hard_failures),
  hard_failures AS quarantine_reasons
FROM STREAM(LIVE.stg_promotions_flagged)
WHERE hard_failures != '';

-- 5. Valid Promotions Stream Setup (Records passing hard checks)
CREATE TEMPORARY STREAMING TABLE stg_promotions_valid AS
SELECT 
  * EXCEPT(err_promo_id, err_discount_neg, err_discount_high, err_dates, warn_name, warn_channel, hard_failures)
FROM STREAM(LIVE.stg_promotions_flagged)
WHERE hard_failures = '';

-- 6. Target Silver Table Declaration & SCD Type 2 Implementation
CREATE STREAMING TABLE silver_promotions
COMMENT "Cleaned promotions data with warn-level data quality flags. Managed as an SCD Type 2 Dimension.";

APPLY CHANGES INTO LIVE.silver_promotions
FROM STREAM(LIVE.stg_promotions_valid)
KEYS (promotion_id)
SEQUENCE BY ingested_timestamp
STORED AS SCD TYPE 2;